# Environment Setup

I used this model to test out building and training the model. I used colab's GPUs.

### Packages

In [ ]:
# %pip install transformers h5py pandas pyarrow protobuf

In [ ]:
# %pip install torch

# install cuda on windows
# %pip install torch --index-url https://download.pytorch.org/whl/cu126

### Mount drive for accessing data in Colab

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Add path so imports work
import sys
sys.path.insert(0, "/content/drive/MyDrive/capstone/Colab resources/style-prompt-generator")

Mounted at /content/drive


# Training Model

CUDA device

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


Constants

In [3]:
H5_PATH     = "/content/drive/MyDrive/capstone/Colab resources/data/merged_audio_500.h5" # <--- UPDATE THIS PATH
META_PATH   = "/content/drive/MyDrive/capstone/Colab resources/data/merged_metadata_500.parquet" # <--- UPDATE THIS PATH
# H5_PATH     = "../data_TEMP/merged_audio_500.h5" # <--- UPDATE THIS PATH
# META_PATH   = "../data_TEMP/merged_metadata_500.parquet" # <--- UPDATE THIS PATH
BATCH_SIZE = 1      # Reduced batch size for memory optimization. Try 1 first. <--- MODIFIED
SAMPLE_RATE = 16_000
MAX_NUM_TURNS = 5

Dataset + loader

In [4]:

from ConvoStyleDataset import ConvoStyleDataset, collate_pad

dataset = ConvoStyleDataset(
    h5_path=H5_PATH,
    meta_path=META_PATH,
    meta_columns=["transcription"],
    sample_rate=SAMPLE_RATE,
    num_turns=MAX_NUM_TURNS # Changed from 5 to 1 to allow loading single-turn utterances for diagnosis
    # max_len_sec=10.0, # Uncomment and adjust this value (e.g., 10 seconds) for further memory optimization.
    # no max_len_sec -- collate_pad handles variable lengths
)

from torch.utils.data import DataLoader
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_pad,
    num_workers=0,
)

Pretrained embedding models

In [ ]:
from transformers import BertTokenizer, BertModel, WavLMModel, Wav2Vec2FeatureExtractor

bert_tokenizer  = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model      = BertModel.from_pretrained("bert-base-uncased").to(device)  # using half() to limit gpu memory for quick tests

# WavLM only needs a feature extractor, not a full processor with tokenizer
wavlm_processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wavlm_model     = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").to(device)

for model in (bert_model, wavlm_model):
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("Encoders loaded and frozen")